# c-GC* Effectome Sweep

This notebook mirrors the `c-GC-star` simulation template, but runs on effectome window artifacts.
It uses the full-conditioning `fcgc` variant and saves per-depth matrix stacks plus summary tables
under `outputs/notebooks/c-GC-star/`.


In [ ]:
from __future__ import annotations

import sys

import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / 'src' / 'effectome').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate effectome project root')

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from effectome.core import CausalisedGC
from effectome.experiments import find_project_root, plot_depth_summary, summarize_stack
from effectome.utils.io import load_artifact, save_artifact, save_matrices

PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')


In [ ]:
METHOD = 'fcgc'
WINDOWS_PATH = PROJECT_ROOT / 'outputs' / 'artifacts' / 'windows.pkl'
N_PASTS = list(range(1, 8))
N_LAGS = 1
N_PERM = 200
ALPHA = 0.01
BETA = 0.001
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebooks' / 'c-GC-star'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Method: {METHOD}')
print(f'Windows path: {WINDOWS_PATH}')
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
windows = load_artifact(WINDOWS_PATH)
print(f'Loaded {windows.n_windows} windows with shape {windows.segments.shape}')


def estimate_cgc_stack(windows, n_past: int) -> np.ndarray:
    matrices = []
    for segment in tqdm(windows.segments, desc=f'c-GC* depth={n_past}'):
        estimator = CausalisedGC(
            n_perm=N_PERM,
            n_pasts=n_past,
            n_lags=N_LAGS,
            method=METHOD,
            signed=True,
        )
        estimator.fit(np.asarray(segment, dtype=np.float64).T, verbose=0)
        matrices.append(
            estimator.get_connectivity_matrix(simulation=True, alpha=ALPHA, beta=BETA)
        )
    return np.asarray(matrices, dtype=np.float32)


In [ ]:
results = {}
summary_rows = []

for n_past in N_PASTS:
    matrices = estimate_cgc_stack(windows, n_past)
    save_matrices(matrices, OUTPUT_DIR / f'cgc_star_depth_{n_past}.npz')
    results[n_past] = matrices
    stats = summarize_stack(matrices)
    stats['n_past'] = n_past
    summary_rows.append(stats)

summary_df = pd.DataFrame(summary_rows).sort_values('n_past').reset_index(drop=True)
summary_df.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
summary_df


In [ ]:
fig = plot_depth_summary(summary_df, 'n_past', 'c-GC* Effectome Summary vs Conditioning Depth')
fig.savefig(OUTPUT_DIR / 'cgc_star_depth_summary.png', dpi=150, bbox_inches='tight')
plt.show()

save_artifact(
    {'method': METHOD, 'depths': N_PASTS, 'summary': summary_df.to_dict(orient='records')},
    OUTPUT_DIR / 'summary.pkl',
)
print('Saved notebook outputs to', OUTPUT_DIR)
